# 3. BaseTool — Subclassing for Full Control

The **most powerful** (and most manual) way to make a tool: subclass `BaseTool`. You reach for this
when a decorator or `from_function` isn't flexible enough.

---

## 1. Simple Definition

> **Kid version:** In tool_decorator and structured_tool notebook you used pre-made tool-makers (a sticker, a labeler). Subclassing
> `BaseTool` is like **building the tool from scratch in your own workshop** — you decide exactly how
> every part behaves. More work, but total control.

**Professional definition:** `BaseTool` is the abstract base class every tool inherits from. By
subclassing it and implementing `_run` (and optionally `_arun`), you define a tool with full control
over its schema, execution, state, and lifecycle.

```python
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

class MultiplyInput(BaseModel):
    a: int = Field(description="first number")
    b: int = Field(description="second number")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers together."
    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

tool = MultiplyTool()
tool.invoke({"a": 6, "b": 7})   # 42
```

---

## 2. Why Does It Exist?

**The problem:** `@tool` and `StructuredTool` cover ~90% of cases, but sometimes you need:
- **State** — the tool holds a resource (a DB connection, an API client, a counter).
- **Custom logic** in construction, validation, or how args are parsed.
- **Both sync and async** with shared setup.
- To integrate a tool into a class hierarchy / reuse behavior across many tools.

### Before (function-based tools can't easily hold state)

```python
@tool
def query_db(sql: str) -> str:
    conn = create_connection()   # ❌ new connection on EVERY call — wasteful
    return conn.execute(sql)
```

### After (subclass holds the connection)

```python
class DBQueryTool(BaseTool):
    name: str = "query_db"
    description: str = "Run a SQL query."
    db_conn: object                     # ← reused state

    def _run(self, sql: str) -> str:
        return self.db_conn.execute(sql)

tool = DBQueryTool(db_conn=create_connection())   # created ONCE, reused
```

The subclass can carry attributes (like `db_conn`) that persist across calls.

---

## 3. Real-Life Analogy

**Hiring a specialist with their own equipment** 🧑‍🔧. A function-tool is a temp worker who brings
nothing. A `BaseTool` subclass is a specialist who **arrives with their own toolkit and workspace**
(state) already set up — so every job is faster and they can do things a temp can't.

---

## 4. Where It Fits in LangChain Architecture

```
Runnable
    │
    ▼
BaseTool                       ← abstract base of ALL tools
    ├── StructuredTool         
    ├── Tool
    └── YourCustomTool         ← you implement _run / _arun
```

Everything — `@tool`, `StructuredTool`, `Tool` — ultimately *is a* `BaseTool`. Subclassing puts you at
the base level with maximum flexibility. Because `BaseTool` is a `Runnable`, tools get `.invoke()`,
`.ainvoke()`, `.batch()`, streaming, and LCEL composition for free.

---

## 5. Internal Working

```
  tool.invoke({"a": 6, "b": 7})
        │
        ▼
  BaseTool.invoke  →  parse & VALIDATE args against args_schema (Pydantic)
        │
        ▼
  calls YOUR _run(a=6, b=7)      (or _arun for async via .ainvoke)
        │
        ▼
  returns the result (optionally wrapped based on response_format)
```

You only write `_run`; `BaseTool` handles validation, the `Runnable` interface, callbacks, and error
plumbing.

---

## 6. What you implement / configure

### `name` (attribute)

**Definition:** The tool's identifier the model uses to call it.

**Why it exists:** Required — the model references the tool by name.

```python
class MyTool(BaseTool):
    name: str = "search"
```

---

### `description` (attribute)

**Definition:** Human-readable purpose; sent to the model.

**Why it exists:** The model's main signal for *when* to use the tool.

```python
    description: str = "Search the knowledge base. Use for policy questions."
```

---

### `args_schema` (attribute)

**Definition:** A Pydantic model defining the tool's arguments.

**Why it exists:** Declares/validates the inputs the model must provide.

**When developers use it:** Whenever the tool takes structured arguments (almost always).

```python
    args_schema: Type[BaseModel] = SearchInput
```

---

### `_run` (method) — REQUIRED

**Definition:** The actual synchronous logic executed when the tool is called.

**Why it exists:** This is the tool's *body* — what it does.

**Real-life use case:** The specialist actually doing the job.

```python
    def _run(self, query: str) -> str:
        return self._search_backend(query)
```

---

### `_arun` (method) — optional async

**Definition:** The asynchronous implementation, used by `.ainvoke()` / async agents.

**Why it exists:** For I/O-bound tools (HTTP, DB) that benefit from async.

**When developers use it:** In async apps; otherwise LangChain can run `_run` in a threadpool.

```python
    async def _arun(self, query: str) -> str:
        return await self._search_backend_async(query)
```

---

### `custom state` (extra attributes)

**Definition:** Any additional fields your tool needs (clients, config, counters).

**Why it exists:** Reuse expensive resources across calls; the whole reason to subclass.

```python
class WeatherTool(BaseTool):
    name: str = "get_weather"
    description: str = "Get current weather for a city."
    api_key: str                       # ← injected once
    args_schema: Type[BaseModel] = WeatherInput

    def _run(self, city: str) -> str:
        return call_weather_api(city, self.api_key)

tool = WeatherTool(api_key="sk-...")   # state set at construction
```

---

## 7. Accessing run context (callbacks / config)

`_run` can accept a `run_manager` for callbacks (logging, streaming progress from inside a tool):

```python
from langchain_core.callbacks import CallbackManagerForToolRun

class MyTool(BaseTool):
    name: str = "long_task"
    description: str = "Run a long task."

    def _run(self, x: str, run_manager: CallbackManagerForToolRun | None = None) -> str:
        if run_manager:
            run_manager.on_text("starting...")
        return do_work(x)
```

---

## Full example

```python
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

class SearchInput(BaseModel):
    query: str = Field(description="the search query")
    top_k: int = Field(default=3, description="number of results")

class KnowledgeBaseTool(BaseTool):
    name: str = "kb_search"
    description: str = "Search the internal knowledge base. Use for product/policy questions."
    args_schema: Type[BaseModel] = SearchInput
    index: object                       # a vector store / search client (state)

    def _run(self, query: str, top_k: int = 3) -> str:
        hits = self.index.search(query, k=top_k)
        return "\n".join(h.text for h in hits)

    async def _arun(self, query: str, top_k: int = 3) -> str:
        hits = await self.index.asearch(query, k=top_k)
        return "\n".join(h.text for h in hits)

tool = KnowledgeBaseTool(index=my_vectorstore)
```

---

## When to subclass vs. use @tool

| Need | Use |
|------|-----|
| Simple function → tool | `@tool`  |
| Third-party / dynamic function | `StructuredTool.from_function`  |
| **Persistent state** (DB/API client) | **subclass `BaseTool`** |
| **Custom validation / lifecycle** | **subclass `BaseTool`** |
| Reuse behavior across many tools | **subclass `BaseTool`** |

> 90% of the time `@tool` is enough. Subclass only when you genuinely need state or control — don't
> over-engineer.

In [1]:
from langchain_core.tools import BaseTool
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import Type

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

d:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# step 1: define the tool with schema
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "multiply two numbers"
    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b:int)->int:
        return a*b

multiply_tool = MultiplyTool()

C:\Users\Mr. Sachin Kapoor\AppData\Local\Temp\ipykernel_29672\100554012.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
C:\Users\Mr. Sachin Kapoor\AppData\Local\Temp\ipykernel_29672\100554012.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [3]:
print("=" * 40)
print("🛠️  Tool Information")
print("=" * 40)
print(f"Name        : {multiply_tool.name}")
print(f"Description : {multiply_tool.description}")
print(f"Arguments   : {multiply_tool.args}")
print("=" * 40)

🛠️  Tool Information
Name        : multiply
Description : multiply two numbers
Arguments   : {'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


In [4]:
multiply_tool.args_schema.model_json_schema()

{'properties': {'a': {'description': 'The first number to add',
   'required': True,
   'title': 'A',
   'type': 'integer'},
  'b': {'description': 'The second number to add',
   'required': True,
   'title': 'B',
   'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'MultiplyInput',
 'type': 'object'}

In [5]:
multiply_tool.invoke({"a": 5, "b": 3})

15

In [6]:
#step2: bind the tool to the language model
llm_with_tools = llm.bind_tools([multiply_tool])

In [7]:
# step3: tool calling
msg = llm_with_tools.invoke("What's 12 times 7?")
msg

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-08T14:55:35.8418853Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14377893500, 'load_duration': 115236200, 'prompt_eval_count': 158, 'prompt_eval_duration': 432495000, 'eval_count': 181, 'eval_duration': 13790708000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019fe1df-1576-73e0-83fd-f94959e5ae7c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 12, 'b': 7}, 'id': '38c61da0-a19d-47b6-b53f-cbf8ad3ccea1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 181, 'total_tokens': 339})

In [8]:
msg.tool_calls

[{'name': 'multiply',
  'args': {'a': 12, 'b': 7},
  'id': '38c61da0-a19d-47b6-b53f-cbf8ad3ccea1',
  'type': 'tool_call'}]

In [9]:
# step 4: tool execution
result = multiply_tool.invoke(llm_with_tools.invoke("can you multiply 3 with 10").tool_calls[0]['args'])
result

30

In [10]:
from typing import Type
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool
from langchain_ollama import ChatOllama

In [11]:
# STEP 1: Define input schemas
class AddInput(BaseModel):
    a: float = Field(description="First number")
    b: float = Field(description="Second number")


class SubtractInput(BaseModel):
    a: float = Field(description="First number")
    b: float = Field(description="Second number")


class MultiplyInput(BaseModel):
    a: float = Field(description="First number")
    b: float = Field(description="Second number")


class DivideInput(BaseModel):
    a: float = Field(description="Numerator")
    b: float = Field(description="Denominator")

In [12]:
# STEP 2: Create BaseTool classes
class AddTool(BaseTool):
    name: str = "add"
    description: str = "Add two numbers."
    args_schema: Type[BaseModel] = AddInput

    def _run(self, a: float, b: float) -> float:
        return a + b


class SubtractTool(BaseTool):
    name: str = "subtract"
    description: str = "Subtract b from a."
    args_schema: Type[BaseModel] = SubtractInput

    def _run(self, a: float, b: float) -> float:
        return a - b


class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers."
    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: float, b: float) -> float:
        return a * b


class DivideTool(BaseTool):
    name: str = "divide"
    description: str = "Divide a by b."
    args_schema: Type[BaseModel] = DivideInput

    def _run(self, a: float, b: float) -> float:
        if b == 0:
            return "Cannot divide by zero"

In [13]:
# STEP 3: Create tool instances
add_tool = AddTool()
subtract_tool = SubtractTool()
multiply_tool = MultiplyTool()
divide_tool = DivideTool()

In [14]:
tools = [add_tool, subtract_tool, multiply_tool, divide_tool]

llm = ChatOllama(model="qwen3:8b")

In [15]:
# STEP 4: Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

In [16]:
for tool in tools:
    print("=" * 50)
    print(f"Tool       : {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Arguments  : {tool.args}")
    print("Schema JSON:")
    print(tool.args_schema.model_json_schema())
    print("=" * 50)

Tool       : add
Description: Add two numbers.
Arguments  : {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}
Schema JSON:
{'properties': {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'AddInput', 'type': 'object'}
Tool       : subtract
Description: Subtract b from a.
Arguments  : {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}
Schema JSON:
{'properties': {'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'SubtractInput', 'type': 'object'}
Tool       : multiply
Description: Multiply two numbers.
Arguments  : {'a': {'description': 'First number', 'title':

In [17]:
# STEP 5: Invoke the LLM with tools
response = llm_with_tools.invoke("What is 25 multiplied by 12?")
response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-08T14:55:57.6302494Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13063625500, 'load_duration': 107751800, 'prompt_eval_count': 313, 'prompt_eval_duration': 87688000, 'eval_count': 155, 'eval_duration': 12845973000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019fe1df-6fb5-7303-b28c-5ebd42b009ed-0', tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 12}, 'id': '804011c2-3eb7-492a-82dc-e7686a577239', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 313, 'output_tokens': 155, 'total_tokens': 468})

In [18]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 25, 'b': 12},
  'id': '804011c2-3eb7-492a-82dc-e7686a577239',
  'type': 'tool_call'}]

In [19]:
print("LLM requested:")
print(response.tool_calls)

# Execute the requested tool
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
tool_args = tool_call["args"]

for tool in tools:
    if tool.name == tool_name:
        tool_result = tool.invoke(tool_args)
        break

print("\nTool result:")
print(tool_result)

LLM requested:
[{'name': 'multiply', 'args': {'a': 25, 'b': 12}, 'id': '804011c2-3eb7-492a-82dc-e7686a577239', 'type': 'tool_call'}]

Tool result:
300.0
